# 1.09 卷积神经网络（CNN）

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.09_cnn.ipynb)


## 为什么要学

机器人要"看"，就需要视觉编码器：

- **ACT** 用 ResNet-18 提取每个相机的特征图；
- **Diffusion Policy** 用 ResNet-18，并把 BatchNorm 换成 GroupNorm，把全局平均池化换成**空间 softmax**；
- π0 用的是 ViT（第 4 章），但 ViT 的第一步"patch embedding"本质上就是一个步长等于核大小的卷积。

参考：[d2l 第 6 章 卷积神经网络](https://zh.d2l.ai/chapter_convolutional-neural-networks/index.html)、[d2l 7.6 残差网络](https://zh.d2l.ai/chapter_convolutional-modern/resnet.html)。

---

## 1. 卷积运算

用一个小窗口（卷积核）在图像上滑动。每个位置做"对应元素相乘再求和"：

$$
Y[i,j] = \sum_{a=0}^{k-1}\sum_{b=0}^{k-1} X[i+a,\, j+b]\cdot K[a,b] + b
$$

（严格来说这叫"互相关"，深度学习里统称卷积。）

两个关键性质让卷积特别适合图像：

1. **局部连接**：每个输出只看一小块邻域；
2. **权重共享**：同一个卷积核在所有位置复用。所以一只"杯子"出现在图像左边还是右边，都能被同一个核检测到（平移等变性）。

### 练习 1：手写二维卷积（单通道）

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

def conv2d_single(X, K):
    """X: (H, W)，K: (kh, kw)，无 padding、步长 1 -> (H-kh+1, W-kw+1)"""
    # TODO: 两重 for 循环，Y[i, j] = (X[i:i+kh, j:j+kw] * K).sum()
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def conv2d_single(X, K):
    """X: (H, W)，K: (kh, kw)，无 padding、步长 1 -> (H-kh+1, W-kw+1)"""
    kh, kw = K.shape
    H, W = X.shape
    Y = torch.zeros(H - kh + 1, W - kw + 1)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i : i + kh, j : j + kw] * K).sum()
    return Y

In [ ]:
# ✅ 自测：和 F.conv2d 对比
X = torch.randn(8, 8); K = torch.randn(3, 3)
ref = F.conv2d(X[None, None], K[None, None])[0, 0]
assert torch.allclose(conv2d_single(X, K), ref, atol=1e-5)
print("通过 ✅")

### 卷积核能检测什么？

In [ ]:
img = torch.zeros(32, 32); img[8:24, 8:24] = 1.0          # 一个白色方块
edge_x = torch.tensor([[-1.0, 0, 1], [-2, 0, 2], [-1, 0, 1]])   # Sobel 算子：检测竖直边缘
edge_y = edge_x.T
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("input")
ax[1].imshow(conv2d_single(img, edge_x), cmap="bwr"); ax[1].set_title("vertical edges")
ax[2].imshow(conv2d_single(img, edge_y), cmap="bwr"); ax[2].set_title("horizontal edges")
for a in ax: a.axis("off")
plt.show()

CNN 的卷积核**不是人设计的，而是学出来的**。浅层学到边缘、颜色，深层学到"把手""杯口"这样的部件。

---

## 2. 填充、步长、多通道

输出尺寸公式（**必须会算**）：

$$
H_{out} = \left\lfloor\frac{H_{in} + 2p - k}{s}\right\rfloor + 1
$$

- **padding $p$**：在边缘补零，保持尺寸（$k=3, p=1, s=1$ 时尺寸不变）；
- **stride $s$**：步长为 2 时尺寸减半；
- **多通道**：输入 $C_{in}$ 个通道、输出 $C_{out}$ 个通道，卷积核形状 `(C_out, C_in, k, k)`。

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=1)
x = torch.randn(8, 3, 96, 96)          # (B, C, H, W)
print(conv(x).shape)                   # (8, 16, 48, 48)
print("权重形状:", conv.weight.shape)   # (16, 3, 3, 3)

# ViT 的 patch embedding = 核大小 = 步长 = patch 大小 的卷积
patch_embed = nn.Conv2d(3, 768, kernel_size=16, stride=16)
print(patch_embed(torch.randn(1, 3, 224, 224)).flatten(2).transpose(1, 2).shape)   # (1, 196, 768)：196 个 token

### 练习 2：计算输出尺寸

In [ ]:
def conv_out_size(H, k, s=1, p=0):
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def conv_out_size(H, k, s=1, p=0):
    return (H + 2 * p - k) // s + 1

In [ ]:
# ✅ 自测：ResNet 的 stem：7x7 卷积 s=2 p=3，再 3x3 最大池化 s=2 p=1
h = conv_out_size(224, 7, 2, 3); h = conv_out_size(h, 3, 2, 1)
assert h == 56
for H, k, s, p in [(96, 3, 2, 1), (48, 5, 1, 2), (17, 4, 3, 0)]:
    assert conv_out_size(H, k, s, p) == nn.Conv2d(1, 1, k, s, p)(torch.zeros(1, 1, H, H)).shape[-1]
print("通过 ✅ ResNet stem: 224 ->", h)

---

## 3. 池化与残差连接

- **最大池化/平均池化**：在窗口里取最大值/平均值，降低分辨率，增加感受野。
- **全局平均池化**：把整个特征图平均成一个向量 `(B, C, H, W) → (B, C)`。

### 残差块（ResNet）

网络很深时，梯度很难传到前面的层。ResNet 的解决方法是让每一块学习**残差**：

$$
\mathbf{y} = \mathbf{x} + F(\mathbf{x})
$$

即使 $F$ 学得不好，至少还有恒等映射 $\mathbf{x}$ 这条"高速公路"，梯度可以直接传回去。**Transformer 的每一层也是残差结构**，这是所有深层网络能训练的关键。

In [ ]:
class ResBlock(nn.Module):
    """ResNet 的基本块：两个 3x3 卷积 + 跳跃连接。"""

    def __init__(self, c_in, c_out, stride=1, norm="batch"):
        super().__init__()
        Norm = (lambda c: nn.BatchNorm2d(c)) if norm == "batch" else (lambda c: nn.GroupNorm(8, c))
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.n1 = Norm(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.n2 = Norm(c_out)
        # 输入输出形状不同时，跳跃连接也要变换一下
        self.skip = nn.Identity() if (stride == 1 and c_in == c_out) else nn.Sequential(
            nn.Conv2d(c_in, c_out, 1, stride, bias=False), Norm(c_out))

    def forward(self, x):
        h = F.relu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))
        return F.relu(h + self.skip(x))

blk = ResBlock(16, 32, stride=2, norm="group")
print(blk(torch.randn(2, 16, 48, 48)).shape)

> **为什么 Diffusion Policy 用 GroupNorm 替换 BatchNorm？** BatchNorm 依赖 batch 统计量，推理时用训练期间的滑动平均。在机器人策略里，训练和部署的数据分布不同、推理 batch=1，再加上 EMA 权重，BatchNorm 容易出问题。GroupNorm 对每个样本独立计算，没有这个问题。

---

## 4. 空间 Softmax：从特征图得到"关键点坐标"

对机器人来说，"物体**在哪里**"比"图里**有什么**"更重要。全局平均池化会丢掉位置信息。**空间 softmax**（Levine 等人 2016 年提出，Diffusion Policy 沿用）对每个通道：

1. 在 $H\times W$ 上做 softmax，得到一个"注意力热力图"；
2. 用它对像素坐标加权平均，得到这个通道关注的**期望坐标** $(x, y)$。

$$
p_c(i,j) = \frac{e^{f_c(i,j)}}{\sum_{i',j'}e^{f_c(i',j')}},\qquad
\hat x_c = \sum_{i,j}p_c(i,j)\,x_j,\quad \hat y_c = \sum_{i,j}p_c(i,j)\,y_i
$$

输出 `(B, C, H, W) → (B, C, 2)`：每个通道变成一个"关键点"。

### 练习 3：实现空间 softmax

In [ ]:
def spatial_softmax(feat):
    """feat: (B, C, H, W) -> (B, C, 2)，最后一维是 (x, y)，坐标范围 [-1, 1]"""
    B, C, H, W = feat.shape
    # TODO:
    # 1) 把 feat 变成 (B, C, H*W) 后在最后一维做 softmax
    # 2) 构造像素坐标：xs = linspace(-1, 1, W)，ys = linspace(-1, 1, H)，用 meshgrid 得到每个像素的 (x, y)
    #    （注意 device=feat.device，否则在 GPU 上会报错）
    # 3) 加权平均
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def spatial_softmax(feat):
    """feat: (B, C, H, W) -> (B, C, 2)，最后一维是 (x, y)，坐标范围 [-1, 1]"""
    B, C, H, W = feat.shape
    attn = torch.softmax(feat.reshape(B, C, H * W), dim=-1)          # (B, C, HW)
    lin = lambda n: torch.linspace(-1, 1, n, device=feat.device)        # 坐标张量要和特征在同一个设备上
    ys, xs = torch.meshgrid(lin(H), lin(W), indexing="ij")
    coords = torch.stack([xs.reshape(-1), ys.reshape(-1)], dim=-1)   # (HW, 2)
    return attn @ coords                                              # (B, C, HW) @ (HW, 2) -> (B, C, 2)

In [ ]:
# ✅ 自测：一个只有一个亮点的特征图，空间 softmax 应该精确找到它的坐标
feat = torch.full((1, 1, 9, 9), -1e4)
feat[0, 0, 2, 6] = 0.0                         # 第 2 行、第 6 列
kp = spatial_softmax(feat)
want = torch.tensor([np.linspace(-1, 1, 9)[6], np.linspace(-1, 1, 9)[2]], dtype=torch.float32)
assert kp.shape == (1, 1, 2) and torch.allclose(kp[0, 0], want, atol=1e-4)
print("通过 ✅ 关键点坐标", kp[0, 0])

---

## 5. 动手：用 CNN 从图像里定位方块

我们用第 7 章会用到的玩具桌面环境生成图像，训练一个小 CNN + 空间 softmax，**直接从像素回归目标方块的坐标**。这正是视觉运动策略里视觉编码器的核心能力。

In [ ]:
import sys
from vla_ladder.envs import ToyTableEnv

env = ToyTableEnv(num_envs=2000, n_blocks=1, seed=0)   # 每张图只有 1 个方块
obs = env.reset()
imgs = torch.tensor(obs["image"])                     # (2000, 3, 48, 48)
targets = torch.tensor(env.blocks[:, 0], dtype=torch.float32)   # (2000, 2) 方块中心坐标
print(imgs.shape, targets.shape)

class KeypointNet(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, c, 3, 1, 1), nn.ReLU(), ResBlock(c, c, norm="group"), ResBlock(c, c, norm="group"),
            nn.Conv2d(c, 8, 1))                       # 8 个关键点通道
        self.head = nn.Linear(8 * 2, 2)

    def forward(self, x):
        kp = spatial_softmax(self.body(x))            # (B, 8, 2)
        return self.head(kp.flatten(1))

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
net = KeypointNet().to(device)
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
X_tr, Y_tr = imgs[:1800].to(device), targets[:1800].to(device)
X_te, Y_te = imgs[1800:].to(device), targets[1800:].to(device)
for step in range(400):
    idx = torch.randint(0, 1800, (64,), device=device)
    loss = F.mse_loss(net(X_tr[idx]), Y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(f"step {step}  loss {loss.item():.4f}")
with torch.no_grad():
    err = (net(X_te) - Y_te).norm(dim=-1).mean().item()
print(f"测试集平均定位误差: {err:.3f}（坐标范围 [-1,1]，一个像素约 {2/48:.3f}）")

---

## 小结

- 卷积 = 局部连接 + 权重共享；输出尺寸 $\lfloor (H+2p-k)/s\rfloor+1$。
- 残差连接 $\mathbf{y}=\mathbf{x}+F(\mathbf{x})$ 让深层网络可训练，Transformer 也依赖它。
- 机器人策略里用 GroupNorm 替代 BatchNorm 更稳定；用空间 softmax 保留"位置"信息。
- ViT 的 patch embedding = 步长等于核大小的卷积。